# ПЗ 7 — Распознавание объектов через Qwen VL

Отправляем кадры в мультимодальную модель Qwen, получаем описание сцены в формате JSON.

In [ ]:
!pip install openai python-dotenv -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

os.makedirs(RESULTS_DIR, exist_ok=True)

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'кадров: {len(frames)}')


In [ ]:
# читаем ключ из .env файла в Drive
env_path = '/content/drive/MyDrive/cv-frames/.env'

with open(env_path) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, _, val = line.partition('=')
            os.environ[key.strip()] = val.strip()

QWEN_API_KEY = os.environ.get('QWEN_API_KEY', '')
print(f'ключ загружен: {QWEN_API_KEY[:8]}...')


In [ ]:
import json
import base64
from openai import OpenAI
from tqdm.notebook import tqdm

client = OpenAI(
    api_key=QWEN_API_KEY,
    base_url='https://dashscope.aliyuncs.com/compatible-mode/v1',
)

def describe_frame(image_path):
    with open(image_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = client.chat.completions.create(
        model='qwen-vl-plus',
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
            {'type': 'text', 'text': 'Опиши объекты на кадре. Верни JSON: {"objects": [...], "scene": "...", "mood": "..."}'},
        ]}],
        max_tokens=512,
    )
    try:
        text = resp.choices[0].message.content.strip().lstrip('```json').rstrip('```')
        return json.loads(text)
    except Exception:
        return {'raw': resp.choices[0].message.content}

results = []
for fname in tqdm(frames[:10], desc='qwen'):
    res = describe_frame(f'{FRAMES_DIR}/{fname}')
    res['frame'] = fname
    results.append(res)
    print(f'{fname}: {res}')

with open(f'{RESULTS_DIR}/llm_detections.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'обработано: {len(results)} кадров')
